<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="https://www.uoc.edu/content/dam/news/images/noticies/2016/202-nova-marca-uoc.jpg" align="left" width="45%">
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">M2.878 · Trabajo de Fin de Máster · <i>XGBoost</i></p>
<p style="margin: 0; text-align:right;">2025-2 · Máster universitario en Ciencia de datos</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Marcos Rodríguez Soler</p>
</div>
</div>
<div style="width:100%;">&nbsp;</div>

# Preprocesamiento de los datos para el modelo _XGBoost_

En el presente _notebook_ se lleva a cabo el entrenamiento del modelo **XGBoost** tanto para el conjunto de datos entero como para los múltiples _clusters_ del _dataset_ global. El flujo de trabajo es similar al de los otros modelos, consistiendo en el cálculo de un conjunto de variables de ventana móvil para capturar en cada muestra los patrones de la demanda pasada, seguido de una optimización de los hiperparámetros del modelo. Continuadamente, se procede con el entrenamiento del modelo y la evaluación en el conjunto de prueba. De la misma forma que en el modelo _Random Forest_, los conjuntos de entrenamiento y de prueba se generan para cada producto individualmente para asegurar que los modelos aprendan los patrones de la demanda de cada serie temporal. Adicionalmente, para que las métricas de evaluación del error sean comparables con las de los otros modelos entrenados en el presente trabajo, se sigue manteniendo la partición 80:20 para generar los conjuntos de aprendizaje y de prueba, ya que de esta forma todos los modelos se evalúan con la misma proporción y cantidad de datos de entrenamiento y de prueba.

Adicionalmente, también se ha utilizado un conjunto de validación, el cual representa el 20% final de las series temporales de cada producto que conforman del conjunto de aprendizaje. La ejecución de este algoritmo se compone de distintas iteraciones donde en cada una se entrena un árbol de decisión que trata de corregir los errores cometidos por los árboles anteriores. Dicho esto, al final de cada iteración se evalúa el modelo con este conjunto de validación con el fin de analizar la evolución del rendimiento del modelo a lo largo del entrenamiento. Esto resulta especialmente relevante en este caso, ya que en _XGBoost_ puede configurarse una condición de parada, que en este caso sirve para parar el entrenamiento del modelo cuando el error cometido en el conjunto de validación no mejora pasadas _n_ iteraciones del algoritmo. Contrariamente, el modelo puede seguir reduciendo el error en el conjunto de entrenamiento pero no en el de validación, hecho que produce que el modelo tienda al sobreajuste, que es lo que se intenta prevenir con este enfoque. Es por esto por lo que no se optimiza el número de árboles en este caso, ya que este hiperparámetro se escogerá de forma automática por el modelo durante el entrenamiento en base a esta condición de parada.

Por otro lado, a diferencia del modelo _Random Forest_, _XGBoost_ no ofrece la posibilidad de entrenar un modelo de regresión que prediga múltiples periodos a futuro. Dado que el entrenamiento de un modelo _XGBoost_ con el volumen de datos que contiene el _dataset_ no es demasiado duradero, se ha optado por entrenar tantos modelos como horizontes de los que se disponga, de manera que cada algoritmo predice la demanda de un horizonte concreto. No obstante, utilizar este enfoque implica considerar que los distintos horizontes de la demanda no están correlacionados entre sí. En las matrices de correlación que se obtuvieron en el _notebook_ donde se entrenaron los modelos _Random Forest_, se observó que las correlaciones entre los distintos horizontes eran moderadas exceptuando el _cluster residual_ donde eran muy débiles, de manera que se estaría ignorando este hecho. Sin embargo, se considera que este error es mucho menor que el que se obtendría al entrenar todos los modelos de forma recursiva, por lo que se decide implementar la solución propuesta dada su mayor simplicidad.

En cuanto a la optimización de hiperparámetros, dado que se entrenarán varias decenas de modelos para cada escenario, se considera que la optimización de cada algoritmo individualmente no es adecuada debido al gran consumo de recursos computacionales que conllevaría. Consecuentemente, se ha optado por optimizar una fracción muy reducida de modelos _XGBoost_ espaciados de forma equitativa a lo largo del horizonte de predicción, donde se pretende agregar los resultados para obtener un conjunto de hiperparámetros generalizables. Estos parámetros serán los que se emplearán para el entrenamiento de cada modelo _XGBoost_ en cada horizonte de la demanda. Asimismo, dado que la cantidad de parámetros que conviene optimizar en _XGBoost_ es considerable, se ha optado por utilizar la clase _RandomizedSearchCV_, que permite ajustar un número fijo de combinaciones aleatorias de hiperparámetros, reduciendo el tiempo de búsqueda en comparación con _GridSearchCV_ sin comprometer la calidad de la búsqueda de forma significativa. Adicionalmente, de la misma forma que en el modelo _Random Forest_, la validación cruzada se lleva a cabo mediante la clase _TimeSeriesSplit_ para mantener la estructura temporal de los datos. Dicho esto, a continuación se ofrece una breve descripción de los hiperparámetros más relevantes del algoritmo _XGBoost_ que se optimizarán para cada modelo.

<ul>
    <li><i>max_depth</i> &#8594; Profundidad máxima a la que puede llegar un árbol del algoritmo</li>
    <li><i>learning_rate</i> &#8594; Contribución de cada árbol a la decisión final del modelo</li>
    <li><i>subsample</i> &#8594; Fracción de datos del conjunto de entrenamiento que se emplea para entrenar un árbol del algoritmo</li>
    <li><i>colsample_bytree</i> &#8594; Fracción de variables del conjunto de aprendizaje que se utiliza para entrenar un árbol del modelo</li>
    <li><i>gamma</i> &#8594; Reducción de pérdida mínima para partir un nodo de un árbol</li>
</ul>

<ol style="list-style: none; padding-left: 0;">
    <li>1. <a href="#ej1">Modelo global</a></li>
    <li>2. <a href="#ej2">Modelos por <i>cluster</i></a> <br>
        &nbsp;&nbsp;2.1. <a href="#ej2.1"><i>Cluster top_ventas</i></a> <br>
        &nbsp;&nbsp;2.2. <a href="#ej2.2"><i>Cluster residual</i></a> <br>
        &nbsp;&nbsp;2.3. <a href="#ej2.3"><i>Cluster alta_rotacion</i></a> <br>
        &nbsp;&nbsp;2.4. <a href="#ej2.4"><i>Cluster estandar</i></a> <br>
    </li>
</ol>

In [ ]:
import time
import random

import pandas as pd
import numpy as np
import seaborn as sns

from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

import matplotlib
import matplotlib.pyplot as plt

import statistics
from collections import Counter

from typing import Any, Dict, List

%matplotlib inline

<br><br>A continuación se define la función _**secuenciador()**_ que, de la misma forma que en el modelo _Random Forest_, sirve para calcular las variables de ventana móvil en el pasado, así como para extraer la demanda en el horizonte de predicción establecido para cada muestra. Asimismo, en este caso también se calculan las variables derivadas hasta 28 días en el pasado, y también se computan tantos horizontes de la demanda como el valor máximo del ciclo de aprovisionamiento de cada conjunto de datos.

In [ ]:
# Función para crear las secuencias de un producto
def secuenciador(df: pd.DataFrame, lag: int, horizonte: int, target: str):
    """Se extraen los últimos valores de las ventas así como las ventas futuras y se calcula la media de ventas en
    el pasado así como su desviación estándar
    Argumentos:
        df: pd.DataFrame -> Conjunto de datos que se desea secuenciar
        lag: int -> Número de periodos pasados que se desean emplear
        horizonte: int -> Número de periodos futuros que se desean emplear
        target: str -> Nombre de la variable objetivo

    Devuelve:
        pd.DataFrame -> DataFrame con la serie secuenciada
    """
    x, y, filas = [], [], []
    for i in range(len(df) - lag - horizonte + 1):
        subventana: pd.DataFrame = df.iloc[i : i + lag]
        x.append(subventana[target].tolist())
          
        siguientes: pd.DataFrame = df.iloc[i + lag : i + lag + horizonte]
        y.append(siguientes[target].tolist())

        filas.append(df.iloc[i + lag])

    diccionario_df = {}
    for i in range(1, lag + 1):
        diccionario_df[f"lag_{i}"] = [x_lag[lag - i] for x_lag in x]

    for i in range(1, horizonte + 1):
        diccionario_df[f"hor_{i}"] = [y_hor[i - 1] for y_hor in y]

    for i in range(1, lag + 1):
        diccionario_df[f"media_rolling_{i}"] = [statistics.mean(x_lag[lag - i :]) for x_lag in x]

    for i in range(2, lag + 1):
        diccionario_df[f"std_rolling_{i}"] = [statistics.stdev(x_lag[lag - i :]) for x_lag in x]

    for i in range(1, lag + 1):
        diccionario_df[f"max_rolling_{i}"] = [max(x_lag[lag - i :]) for x_lag in x]
            
    df_lags_hor: pd.DataFrame = pd.DataFrame(diccionario_df)
    df_variables: pd.DataFrame = pd.DataFrame(filas).reset_index(drop=True)

    return pd.concat([df_lags_hor, df_variables], axis=1)

<br><br>
De la misma forma que en **Random Forest**, también se definen las funciones _**rmse()**_ y _**rmsse_producto()**_, que permiten evaluar el error de las predicciones de los modelos.

La función _**rmse()**_ calcula la raíz del error cuadrático medio entre la demanda real y la demanda predicha sobre el conjunto de prueba. Por otro lado, la función _**rmsse_producto()**_ calcula el error cuadrático medio escalado cada producto y un horizonte temporal concreto, normalizando el error respecto a la variabilidad histórica observada en el conjunto de entrenamiento de cada serie. Este enfoque permite comparar el rendimiento de los modelos entre productos con distintos niveles de demanda.

Para la evaluación de los modelos, ambas métricas se calculan inicialmente para cada combinación de producto, $p$, y horizonte temporal, $h$. Posteriormente, para cada producto se promedian únicamente los errores correspondientes a los horizontes comprendidos dentro del ciclo de aprovisionamiento de cada producto, definido como la suma del tiempo de entrega, $L$, y los días entre pedidos, $R$. 

$error_p = \frac{1}{R+L} \sum_{h=1}^{R+L} error_{p,h}$

Finalmente, el rendimiento global de cada modelo se obtiene promediando los errores operativos de todos los productos, $N$.

$error_{modelo} = \frac{1}{N} \sum_{p=1}^{N} error_p$

En el caso del RMSE, esta métrica se utilizará posteriormente en el apartado **Evaluación del Impacto Económico** durante la simulación de inventario para aproximar la incertidumbre asociada a las predicciones y analizar su efecto sobre las decisiones de reposición y los costes operativos del sistema.

In [ ]:
# Funciones para calcular el RMSE y el RMSSE
def rmse(y_real: pd.Series, y_pred: pd.Series) -> float:
    """Devuelve el RMSE de las predicciones de un modelo sobre un conjunto de prueba

    Argumentos:
        y_real: pd.Series -> Demanda real del conjunto de prueba
        y_pred: pd.Series -> Demanda predicha para el conjunto de prueba

    Devuelve
        float -> RMSE
    """
    return np.sqrt(np.mean(np.abs(y_real - y_pred) ** 2))


def rmsse_producto(df: pd.DataFrame, df_test: pd.DataFrame, horizonte: int) -> Dict[str, float]:
    """Devuelve el RMSSE de cada producto para un horizonte temporal concreto

    Argumentos:
        df: pd.DataFrame -> DataFrame completo del conjunto de datos
        df_test: pd.DataFrame -> Resultados del conjunto de prueba
        horizonte: int -> Horizonte temporal

    Devuelve:
        Dict[str, float] -> RMSSE de cada producto en un horizonte temporal concreto
    """
    valores: Dict[str, float] = {}

    for producto, grupo_test in df_test.groupby("producto"):

        y_real: np.ndarray = grupo_test[f"real_{horizonte}"].values
        y_pred: np.ndarray = grupo_test[f"pred_{horizonte}"].values

        numerador: float = np.mean((y_real - y_pred) ** 2)

        # Datos de entrenamiento del producto
        grupo_train: pd.DataFrame = (
            df[df["producto"] == producto]
            .sort_values("fecha")
        )

        PARTICION: int = int(len(grupo_train) * 0.8)

        y_train: np.ndarray = (
            grupo_train[f"hor_{horizonte}"]
            .iloc[:PARTICION]
            .values
        )

        denominador: float = np.mean(
            (y_train[1:] - y_train[:-1]) ** 2
        )

        if denominador != 0 and not np.isnan(denominador):
            valores[producto] = np.sqrt(numerador / denominador)

    return valores

<br><br><a id="ej1"></a>
# 1. Modelo global

En este apartado se entrena el modelo _XGBoost_ con todos los productos del conjunto de datos. Para ello, primero se importan los datos obtenidos del **Análisis Exploratorio de los Datos** y se calculan las variables de ventana móvil hasta 28 días en el pasado para cada muestra. Cabe destacar que en este caso no se visualizan las matrices de correlación ya son las mismas que se representaron en el modelo **Random Forest** tanto para este conjunto de datos global como para cada _cluster_.

In [ ]:
# Se importan los datos preprocesados del análisis exploratorio de datos
ruta_dataset: str = "../AED/dataset_preprocesado.csv"
dataset_global: pd.DataFrame = pd.read_csv(ruta_dataset).drop("Unnamed: 0", axis=1)

In [ ]:
# Se crean las nuevas variables para el conjunto de datos completo para entrenar un modelo global
LAG: int = 28
HORIZONTE: int = max(dataset_global["diasLeadtime"] + dataset_global["diasEntrePedidos"])
dataset_secuenciado: List[pd.DataFrame] = []

for producto, df_producto in dataset_global.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    df_producto_sec: pd.DataFrame = secuenciador(
        df=df_producto,
        lag=LAG,
        horizonte=HORIZONTE,
        target="udsVenta"
    )
    
    df_producto_sec["producto"] = producto
    
    dataset_secuenciado.append(df_producto_sec)

dataset_secuenciado: pd.DataFrame = pd.concat(dataset_secuenciado, ignore_index=True)

<br><br>Para capturar los patrones de la demanda en intervalos acumulativos de una semana, de la misma forma que en _Random Forest_, se escogen únicamente las variables de ventana móvil cada 7 días en el pasado.

In [ ]:
# Se selecciona el subconjunto de variables de interés del conjunto de datos global
dataset_global_subset: pd.DataFrame = (
    dataset_secuenciado[
        [
            "producto", "fecha", "diasEntrePedidos", "diasLeadtime", "eurPrecioMedio", "bolHoliday_reconstruido", "bolOpen_reconstruido",
            "isPromo", "lag_7", "lag_14", "lag_21", "lag_28", "dia_semana_str_Lunes", "dia_semana_str_Martes", "dia_semana_str_Miércoles",
            "dia_semana_str_Jueves", "dia_semana_str_Viernes", "dia_semana_str_Sábado", "dia_semana_str_Domingo", "media_rolling_7",
            "media_rolling_14", "media_rolling_21", "media_rolling_28", "max_rolling_14", "max_rolling_21", "max_rolling_28", "std_rolling_7",
            "std_rolling_14", "std_rolling_21", "std_rolling_28", "udsVenta"
        ] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)] + [f"mes_str_{i}" for i in range(1, 13)]
    ]
)

In [ ]:
# Se crean los conjuntos de entrenamiento y de prueba del conjunto de datos global
horizontes: List[str] = [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

conjuntos_dataset_global: Dict[str, pd.DataFrame] = {}
lista_x_train, lista_x_val, lista_x_test = [], [], []
lista_y_train, lista_y_val, lista_y_test = [], [], []

for producto, df_producto in dataset_global_subset.drop("udsVenta", axis=1).groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").drop("producto", axis=1)
    
    x_producto: pd.DataFrame = df_producto.drop(columns=horizontes + ["fecha"])
    y_producto: pd.DataFrame = df_producto[horizontes]
    
    PARTICION_TRAIN_TEST: int = int(len(df_producto) * 0.8)
    datos_x_train: pd.DataFrame = x_producto.iloc[:PARTICION_TRAIN_TEST]
    lista_x_test.append(x_producto.iloc[PARTICION_TRAIN_TEST:])

    PARTICION_TRAIN_VAL: int = int(len(datos_x_train) * 0.8)
    lista_x_train.append(datos_x_train.iloc[:PARTICION_TRAIN_VAL])
    lista_x_val.append(datos_x_train.iloc[PARTICION_TRAIN_VAL:])
    

    datos_y_train: pd.Series = y_producto.iloc[:PARTICION_TRAIN_TEST]
    lista_y_test.append(y_producto.iloc[PARTICION_TRAIN_TEST:])
    
    lista_y_train.append(datos_y_train.iloc[:PARTICION_TRAIN_VAL])
    lista_y_val.append(datos_y_train.iloc[PARTICION_TRAIN_VAL:])
    

conjuntos_dataset_global["x_train"] = pd.concat(lista_x_train, ignore_index=True)
conjuntos_dataset_global["x_test"] = pd.concat(lista_x_test, ignore_index=True)
conjuntos_dataset_global["x_val"] = pd.concat(lista_x_val, ignore_index=True)

conjuntos_dataset_global["y_train"] = pd.concat(lista_y_train, ignore_index=True)
conjuntos_dataset_global["y_test"]  = pd.concat(lista_y_test, ignore_index=True)
conjuntos_dataset_global["y_val"]  = pd.concat(lista_y_val, ignore_index=True)

print(f"El conjunto x_train contiene {conjuntos_dataset_global["x_train"].shape[0]} filas y {conjuntos_dataset_global["x_train"].shape[1]} columnas")
print(f"El conjunto y_train contiene {conjuntos_dataset_global["y_train"].shape[0]} filas y {conjuntos_dataset_global["y_train"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_test contiene {conjuntos_dataset_global["x_test"].shape[0]} filas y {conjuntos_dataset_global["x_test"].shape[1]} columnas")
print(f"El conjunto y_test contiene {conjuntos_dataset_global["y_test"].shape[0]} filas y {conjuntos_dataset_global["y_test"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_val contiene {conjuntos_dataset_global["x_val"].shape[0]} filas y {conjuntos_dataset_global["x_val"].shape[1]} columnas")
print(f"El conjunto y_val contiene {conjuntos_dataset_global["y_val"].shape[0]} filas y {conjuntos_dataset_global["y_val"].shape[1]} columnas")

In [ ]:
# Se optimizan varios modelos XGBoost a lo largo de todo el horizonte de predicción del dataset global
N_ITER: int = 20
SEMILLA: int = 42
tscv: TimeSeriesSplit = TimeSeriesSplit(n_splits=5)
optimizaciones: List[int] = [i for i in range(1, HORIZONTE + 1, 13)]

param_search_xgboost: Dict[str, List[int | float]] = {
    "max_depth": [5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3]
}

mejores_max_depth: List[int] = []
mejores_learning_rate: List[float] = []
mejores_subsample: List[float] = []
mejores_colsample_bytree: List[float] = []
mejores_gamma: List[float] = []

tiempo_inicial_opt: float = time.time()
for h in optimizaciones:
    modelo: XGBRegressor = XGBRegressor(
        objective="reg:squarederror",
        n_jobs=-1
    )
    
    search_xgboost_global: RandomizedSearchCV = RandomizedSearchCV(
        modelo,
        param_distributions=param_search_xgboost,
        n_iter=N_ITER,
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        verbose=1,
        random_state=SEMILLA
    )

    tiempo_inicial: float = time.time()
    search_xgboost_global.fit(conjuntos_dataset_global["x_train"], conjuntos_dataset_global["y_train"][f"hor_{h}"])
    tiempo_final: float = time.time()
    
    mejores_max_depth.append(search_xgboost_global.best_params_["max_depth"])
    mejores_learning_rate.append(search_xgboost_global.best_params_["learning_rate"])
    mejores_subsample.append(search_xgboost_global.best_params_["subsample"])
    mejores_colsample_bytree.append(search_xgboost_global.best_params_["colsample_bytree"])
    mejores_gamma.append(search_xgboost_global.best_params_["gamma"])

    
    print(
        "La primera iteración de la optimización de hiperparámetros de los modelos globales de XGBoost tardó "
        f"{round(tiempo_final - tiempo_inicial, 2)} segundos en completarse"
    )

tiempo_final_opt: float = time.time()
print("\n")
print(
    "La optimización de hiperparámetros de los modelos globales de XGBoost tardó "
    f"{round(tiempo_final_opt - tiempo_inicial_opt, 2)} segundos en completarse"
)

In [ ]:
# Se calculan los mejores hiperparámetros de las múltiples optimizaciones. En el caso de los parámetros enteros
# se escoge el valor más repetido, y en el caso de los hiperparámetros decimales se realiza la media
def moda(lista: List[int]) -> int:
    """Devuelve el valor más repetido de una lista

    Argumentos:
        lista (List[int]) -> Lista de la que se quiere hallar el valor más repetido

    Devuelve:
        int -> Moda de la lista
    """
    return Counter(lista).most_common(1)[0][0]

max_depth_opt: int = moda(mejores_max_depth)
learning_rate_opt: float = round(np.mean(mejores_learning_rate), 2)
subsample_opt: float = round(np.mean(mejores_subsample), 1)
colsample_bytree_opt: float = round(np.mean(mejores_colsample_bytree), 1)
gamma_opt: float = round(np.mean(mejores_gamma), 2)

print(
    f"La profundidad máxima óptima de los árboles es de max_depth={max_depth_opt}, la mejor tasa de aprendizaje "
    f"es {learning_rate_opt}, la mejor fracción de muestras para entrenar a cada árbol es {subsample_opt}, la mejor "
    f"fracción de variables para entrenar cada árbol es {colsample_bytree_opt}, y la mejor reducción de pérdida mínima"
    f"para partir un nodo es {gamma_opt}"
)

In [ ]:
# Se entrenan los modelos XGBoost para predecir la demanda con los datos del dataset global
EARLY_STOPPING_ROUNDS: int = 10
modelos_globales: Dict[str, XGBRegressor] = {}
tiempo_inicial: int = time.time()

for h in range(1, HORIZONTE + 1):
    modelo_global = XGBRegressor(
        max_depth=max_depth_opt,
        learning_rate=learning_rate_opt,
        subsample=subsample_opt,
        colsample_bytree=colsample_bytree_opt,
        gamma=gamma_opt,
        objective="reg:squarederror",
        eval_metric="rmse",
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=SEMILLA,
        n_jobs=-1
    )
    
    modelo_global.fit(
        conjuntos_dataset_global["x_train"],
        conjuntos_dataset_global["y_train"][f"hor_{h}"],
        eval_set=[(
            conjuntos_dataset_global["x_val"],
            conjuntos_dataset_global["y_val"][f"hor_{h}"]
        )],
        verbose=False
    )
    
    modelos_globales[h] = modelo_global

tiempo_final: int = time.time()
print(f"Los modelos globales tardaron {round(tiempo_final - tiempo_inicial, 2)} segundos en entrenarse")

In [ ]:
# Se entrenan los modelos XGBoost para predecir la demanda con los datos del dataset global
EARLY_STOPPING_ROUNDS: int = 10
modelos_globales: Dict[str, XGBRegressor] = {}
tiempo_inicial: int = time.time()

for h in range(1, HORIZONTE + 1):
    modelo_global = XGBRegressor(
        max_depth=10,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.3,
        objective="reg:squarederror",
        eval_metric="rmse",
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=SEMILLA,
        n_jobs=-1
    )
    
    modelo_global.fit(
        conjuntos_dataset_global["x_train"],
        conjuntos_dataset_global["y_train"][f"hor_{h}"],
        eval_set=[(
            conjuntos_dataset_global["x_val"],
            conjuntos_dataset_global["y_val"][f"hor_{h}"]
        )],
        verbose=False
    )
    
    modelos_globales[h] = modelo_global

tiempo_final: int = time.time()
print(f"Los modelos globales tardaron {round(tiempo_final - tiempo_inicial, 2)} segundos en entrenarse")

In [ ]:
# Se realizan las predicciones en el conjunto de prueba del dataset global
y_pred_global: np.ndarray = np.column_stack([
    modelos_globales[h].predict(conjuntos_dataset_global["x_test"])
    for h in range(1, HORIZONTE + 1)
])

y_pred_global_df: pd.DataFrame = pd.DataFrame(
    y_pred_global,
    columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
)

In [ ]:
# Se asocian a las predicciones y los valores reales los productos y las fechas correspondientes
lista_meta_test: List[pd.DataFrame] = []

for producto, df_producto in dataset_global_subset.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    meta_info_test: pd.DataFrame = df_producto.iloc[PARTICION:][[
        "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
    ]]
    lista_meta_test.append(meta_info_test)

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)

y_real_global_df: pd.DataFrame = pd.DataFrame(
    conjuntos_dataset_global["y_test"].values,
    columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
)

resultados_modelo_global: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_global_df, y_pred_global_df],
    axis=1
)

resultados_modelo_global["fecha"] = pd.to_datetime(resultados_modelo_global["fecha"])

In [ ]:
# Se calculan los errores cometidos en el conjunto de prueba del dataset global
rmses: Dict[str, float] = {
    f"hor_{i}": rmse(
        resultados_modelo_global[f"real_{i}"],
        resultados_modelo_global[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Promedio del RMSE de los horizontes temporales del modelo global")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<br><br> Se calculan los RMSE y RMSSE a nivel de horizonte por producto del modelo global. Los RMSE se exportan en un archivo _csv_ para la **Evaluación del Impacto Económico**, y los RMSSE se usan para obtener el promedio del modelo de esta métrica como se explicó al inicio del _notebook_.

In [ ]:
# Se calculan los RMSE en el conjunto de prueba del dataset global por producto y se exportan para la evaluación económica
rmses_global: List[pd.DataFrame] = []

for producto, df_producto in resultados_modelo_global.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_global.append(rmses_df)

rmses_global: pd.DataFrame = pd.concat(rmses_global, ignore_index=True)
rmses_global.to_csv("rmses_xgboost_global.csv")

In [ ]:
# Se calculan los RMSSE de cada producto en cada horizonte temporal del modelo global
lista_rmsse: List[Dict[str, Any]] = []
for horizonte in range(1, HORIZONTE + 1):

    rmsse_horizonte: Dict[str, float] = rmsse_producto(
        dataset_global_subset,
        resultados_modelo_global,
        horizonte
    )

    for producto, valor in rmsse_horizonte.items():

        lista_rmsse.append({
            "producto": producto,
            "horizonte": horizonte,
            "rmsse": valor
        })

# Se disponen todas las combinaciones de RMSSE de productos y horizontes en un DataFrame
df_rmsses: pd.DataFrame = pd.DataFrame(lista_rmsse)
df_rmsses = df_rmsses.pivot(
    index="producto",
    columns="horizonte",
    values="rmsse"
).reset_index()
df_rmsses.columns = ["producto"] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

# Se calcula el RMSSE de cada producto como el promedio de todos los RMSSE en cada horizonte temporal
# hasta el que corresponde con el ciclo de aprovisionamiento del artículo
rmsse_productos: List[float] = []
for producto, df_producto in resultados_modelo_global.groupby("producto"):

    ciclo_aprov: int = int(
        df_producto["diasLeadtime"].mean() +
        df_producto["diasEntrePedidos"].mean()
    )

    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]

    fila_producto_df = df_rmsses[df_rmsses["producto"] == producto]
    if fila_producto_df.empty:
        continue
    fila_producto: pd.Series = fila_producto_df.iloc[0]

    rmsse_producto_promedio: float = fila_producto[columnas_horizontes].mean()
    rmsse_productos.append(rmsse_producto_promedio)

# Se promedian todos los RMSSEs de los ítems para obtener el RMSSE del modelo global
rmsse_modelo_global: float = np.mean(rmsse_productos)
print(f"El RMSSE promedio del modelo global es {rmsse_modelo_global:.4f}")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios del dataset global
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados_modelo_global["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados_modelo_global[resultados_modelo_global["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} del dataset global")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se exportan los resultados
resultados_modelo_global.to_csv("res_xgboost_global.csv")

<br><br><a id="ej2"></a>
# 2. Modelos por _cluster_

En esta sección del _notebook_ se entrena un conjunto de regresores _XGBoost_ para cada uno de los grupos en los que se ha dividido el conjunto de datos.

<a id="ej2.1"></a>
## 2.1. _Cluster top_ventas_

Inicialmente, se calculan las variables de ventana móvil mediante la función _**secuenciador()**_ del _cluster top_ventas_ para 28 días en el pasado.

In [ ]:
# Se extrae el grupo top_ventas del dataset global
top_ventas: pd.DataFrame = dataset_global[dataset_global["cluster_nombre"] == "top_ventas"]

In [ ]:
# Se crean las nuevas variables para el conjunto de datos del grupo top_ventas
HORIZONTE: int = max(top_ventas["diasLeadtime"] + top_ventas["diasEntrePedidos"])
top_ventas_secuenciado: List[pd.DataFrame] = []

for producto, df_producto in top_ventas.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    df_producto_sec: pd.DataFrame = secuenciador(
        df=df_producto,
        lag=LAG,
        horizonte=HORIZONTE,
        target="udsVenta"
    )
    
    df_producto_sec["producto"] = producto
    
    top_ventas_secuenciado.append(df_producto_sec)

top_ventas_secuenciado: pd.DataFrame = pd.concat(top_ventas_secuenciado, ignore_index=True)

<br><br>Se seleccionan las variables de ventana móvil cada 7 días en el pasado junto con los atributos del _dataset_ original, y se entrena el modelo _XGBoost_.

In [ ]:
# Se selecciona el subconjunto de variables de interés del conjunto de datos de top_ventas
top_ventas_subset: pd.DataFrame = (
    top_ventas_secuenciado[
        [
            "producto", "fecha", "diasEntrePedidos", "diasLeadtime", "eurPrecioMedio", "bolHoliday_reconstruido", "bolOpen_reconstruido",
            "isPromo", "lag_7", "lag_14", "lag_21", "lag_28", "dia_semana_str_Lunes", "dia_semana_str_Martes", "dia_semana_str_Miércoles",
            "dia_semana_str_Jueves", "dia_semana_str_Viernes", "dia_semana_str_Sábado", "dia_semana_str_Domingo", "media_rolling_7",
            "media_rolling_14", "media_rolling_21", "media_rolling_28", "max_rolling_14", "max_rolling_21", "max_rolling_28", "std_rolling_7",
            "std_rolling_14", "std_rolling_21", "std_rolling_28", "udsVenta"
        ] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)] + [f"mes_str_{i}" for i in range(1, 13)]
    ]
)

In [ ]:
# Se crean los conjuntos de entrenamiento y de prueba del cluster top_ventas
horizontes: List[str] = [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

conjuntos_top_ventas: Dict[str, pd.DataFrame] = {}
lista_x_train, lista_x_val, lista_x_test = [], [], []
lista_y_train, lista_y_val, lista_y_test = [], [], []

for producto, df_producto in top_ventas_subset.drop("udsVenta", axis=1).groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").drop("producto", axis=1)
    
    x_producto: pd.DataFrame = df_producto.drop(columns=horizontes + ["fecha"])
    y_producto: pd.DataFrame = df_producto[horizontes]
    
    PARTICION_TRAIN_TEST: int = int(len(df_producto) * 0.8)
    datos_x_train: pd.DataFrame = x_producto.iloc[:PARTICION_TRAIN_TEST]
    lista_x_test.append(x_producto.iloc[PARTICION_TRAIN_TEST:])

    PARTICION_TRAIN_VAL: int = int(len(datos_x_train) * 0.8)
    lista_x_train.append(datos_x_train.iloc[:PARTICION_TRAIN_VAL])
    lista_x_val.append(datos_x_train.iloc[PARTICION_TRAIN_VAL:])
    

    datos_y_train: pd.Series = y_producto.iloc[:PARTICION_TRAIN_TEST]
    lista_y_test.append(y_producto.iloc[PARTICION_TRAIN_TEST:])
    
    lista_y_train.append(datos_y_train.iloc[:PARTICION_TRAIN_VAL])
    lista_y_val.append(datos_y_train.iloc[PARTICION_TRAIN_VAL:])
    

conjuntos_top_ventas["x_train"] = pd.concat(lista_x_train, ignore_index=True)
conjuntos_top_ventas["x_test"] = pd.concat(lista_x_test, ignore_index=True)
conjuntos_top_ventas["x_val"] = pd.concat(lista_x_val, ignore_index=True)

conjuntos_top_ventas["y_train"] = pd.concat(lista_y_train, ignore_index=True)
conjuntos_top_ventas["y_test"]  = pd.concat(lista_y_test, ignore_index=True)
conjuntos_top_ventas["y_val"]  = pd.concat(lista_y_val, ignore_index=True)

print(f"El conjunto x_train contiene {conjuntos_top_ventas["x_train"].shape[0]} filas y {conjuntos_top_ventas["x_train"].shape[1]} columnas")
print(f"El conjunto y_train contiene {conjuntos_top_ventas["y_train"].shape[0]} filas y {conjuntos_top_ventas["y_train"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_test contiene {conjuntos_top_ventas["x_test"].shape[0]} filas y {conjuntos_top_ventas["x_test"].shape[1]} columnas")
print(f"El conjunto y_test contiene {conjuntos_top_ventas["y_test"].shape[0]} filas y {conjuntos_top_ventas["y_test"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_val contiene {conjuntos_top_ventas["x_val"].shape[0]} filas y {conjuntos_top_ventas["x_val"].shape[1]} columnas")
print(f"El conjunto y_val contiene {conjuntos_top_ventas["y_val"].shape[0]} filas y {conjuntos_top_ventas["y_val"].shape[1]} columnas")

In [ ]:
# Se optimizan varios modelos XGBoost a lo largo de todo el horizonte de predicción del cluster top_ventas
optimizaciones: List[int] = [i for i in range(1, HORIZONTE + 1, 7)]

mejores_max_depth: List[int] = []
mejores_learning_rate: List[float] = []
mejores_subsample: List[float] = []
mejores_colsample_bytree: List[float] = []
mejores_gamma: List[float] = []

tiempo_inicial_opt: float = time.time()
for h in optimizaciones:
    modelo: XGBRegressor = XGBRegressor(
        objective="reg:squarederror",
        n_jobs=-1
    )
    
    search_xgboost_top_ventas: RandomizedSearchCV = RandomizedSearchCV(
        modelo,
        param_distributions=param_search_xgboost,
        n_iter=N_ITER,
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        verbose=1,
        random_state=SEMILLA
    )

    tiempo_inicial: float = time.time()
    search_xgboost_top_ventas.fit(conjuntos_top_ventas["x_train"], conjuntos_top_ventas["y_train"][f"hor_{h}"])
    tiempo_final: float = time.time()
    
    mejores_max_depth.append(search_xgboost_top_ventas.best_params_["max_depth"])
    mejores_learning_rate.append(search_xgboost_top_ventas.best_params_["learning_rate"])
    mejores_subsample.append(search_xgboost_top_ventas.best_params_["subsample"])
    mejores_colsample_bytree.append(search_xgboost_top_ventas.best_params_["colsample_bytree"])
    mejores_gamma.append(search_xgboost_top_ventas.best_params_["gamma"])

    
    print(
        "La primera iteración de la optimización de hiperparámetros de los modelos de top_ventas de XGBoost tardó "
        f"{round(tiempo_final - tiempo_inicial, 2)} segundos en completarse"
    )

tiempo_final_opt: float = time.time()
print("\n")
print(
    "La optimización de hiperparámetros de los modelos de top_ventas de XGBoost tardó "
    f"{round(tiempo_final_opt - tiempo_inicial_opt, 2)} segundos en completarse"
)

In [ ]:
# Se calculan los mejores hiperparámetros de las múltiples optimizaciones del cluster top_ventas
max_depth_opt: int = moda(mejores_max_depth)
learning_rate_opt: float = round(np.mean(mejores_learning_rate), 2)
subsample_opt: float = round(np.mean(mejores_subsample), 1)
colsample_bytree_opt: float = round(np.mean(mejores_colsample_bytree), 1)
gamma_opt: float = round(np.mean(mejores_gamma), 2)

print(
    f"La profundidad máxima óptima de los árboles es de max_depth={max_depth_opt}, la mejor tasa de aprendizaje "
    f"es {learning_rate_opt}, la mejor fracción de muestras para entrenar a cada árbol es {subsample_opt}, la mejor "
    f"fracción de variables para entrenar cada árbol es {colsample_bytree_opt}, y la mejor reducción de pérdida mínima"
    f"para partir un nodo es {gamma_opt}"
)

In [ ]:
# Se entrenan los modelos XGBoost para predecir la demanda con los datos del cluster top_ventas
modelos_top_ventas: Dict[str, XGBRegressor] = {}
tiempo_inicial: int = time.time()

for h in range(1, HORIZONTE + 1):
    modelo_top_ventas = XGBRegressor(
        max_depth=max_depth_opt,
        learning_rate=learning_rate_opt,
        subsample=subsample_opt,
        colsample_bytree=colsample_bytree_opt,
        gamma=gamma_opt,
        objective="reg:squarederror",
        eval_metric="rmse",
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=SEMILLA,
        n_jobs=-1
    )
    
    modelo_top_ventas.fit(
        conjuntos_top_ventas["x_train"],
        conjuntos_top_ventas["y_train"][f"hor_{h}"],
        eval_set=[(
            conjuntos_top_ventas["x_val"],
            conjuntos_top_ventas["y_val"][f"hor_{h}"]
        )],
        verbose=False
    )
    
    modelos_top_ventas[h] = modelo_top_ventas

tiempo_final: int = time.time()
print(f"Los modelos top_ventas tardaron {round(tiempo_final - tiempo_inicial, 2)} segundos en entrenarse")

In [ ]:
# Se realizan las predicciones en el conjunto de prueba del dataset de top_ventas
y_pred_top_ventas: np.ndarray = np.column_stack([
    modelos_top_ventas[h].predict(conjuntos_top_ventas["x_test"])
    for h in range(1, HORIZONTE + 1)
])

y_pred_top_ventas_df: pd.DataFrame = pd.DataFrame(
    y_pred_top_ventas,
    columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
)

In [ ]:
# Se asocian a las predicciones y los valores reales los productos y las fechas correspondientes
lista_meta_test: List[pd.DataFrame] = []

for producto, df_producto in top_ventas_subset.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    meta_info_test: pd.DataFrame = df_producto.iloc[PARTICION:][[
        "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
    ]]
    lista_meta_test.append(meta_info_test)

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)

y_real_top_ventas_df: pd.DataFrame = pd.DataFrame(
    conjuntos_top_ventas["y_test"].values,
    columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
)

resultados_top_ventas: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_top_ventas_df, y_pred_top_ventas_df],
    axis=1
)

resultados_top_ventas["fecha"] = pd.to_datetime(resultados_top_ventas["fecha"])

In [ ]:
# Se calculan los errores cometidos en el conjunto de prueba del cluster top_ventas
rmses: Dict[str, float] = {
    f"hor_{i}": rmse(
        resultados_top_ventas[f"real_{i}"],
        resultados_top_ventas[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Promedio del RMSE de los horizontes temporales del modelo top_ventas")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<br><br>Se calculan los RMSE y RMSSE a nivel de horizonte por producto del _cluster top_ventas_. Los RMSE se exportan en un archivo _csv_ para la **Evaluación del Impacto Económico**, y los RMSSE se usan para obtener el promedio del modelo de esta métrica como se explicó al inicio del _notebook_.

In [ ]:
# Se calculan los RMSE en el conjunto de prueba del cluster top_ventas por producto y se exportan para la evaluación económica
rmses_top_ventas: List[pd.DataFrame] = []

for producto, df_producto in resultados_top_ventas.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_top_ventas.append(rmses_df)

rmses_top_ventas: pd.DataFrame = pd.concat(rmses_top_ventas, ignore_index=True)
rmses_top_ventas.to_csv("rmses_xgboost_top_ventas.csv")

In [ ]:
# Se calculan los RMSSE de cada producto en cada horizonte temporal del modelo top_ventas
lista_rmsse: List[Dict[str, Any]] = []
for horizonte in range(1, HORIZONTE + 1):

    rmsse_horizonte: Dict[str, float] = rmsse_producto(
        top_ventas_subset,
        resultados_top_ventas,
        horizonte
    )

    for producto, valor in rmsse_horizonte.items():

        lista_rmsse.append({
            "producto": producto,
            "horizonte": horizonte,
            "rmsse": valor
        })

# Se disponen todas las combinaciones de RMSSE de productos y horizontes en un DataFrame
df_rmsses: pd.DataFrame = pd.DataFrame(lista_rmsse)
df_rmsses = df_rmsses.pivot(
    index="producto",
    columns="horizonte",
    values="rmsse"
).reset_index()
df_rmsses.columns = ["producto"] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

# Se calcula el RMSSE de cada producto como el promedio de todos los RMSSE en cada horizonte temporal
# hasta el que corresponde con el ciclo de aprovisionamiento del artículo
rmsse_productos: List[float] = []
for producto, df_producto in resultados_top_ventas.groupby("producto"):

    ciclo_aprov: int = int(
        df_producto["diasLeadtime"].mean() +
        df_producto["diasEntrePedidos"].mean()
    )

    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]

    fila_producto_df = df_rmsses[df_rmsses["producto"] == producto]
    if fila_producto_df.empty:
        continue
    fila_producto: pd.Series = fila_producto_df.iloc[0]

    rmsse_producto_promedio: float = fila_producto[columnas_horizontes].mean()
    rmsse_productos.append(rmsse_producto_promedio)

# Se promedian todos los RMSSEs de los ítems para obtener el RMSSE del modelo top_ventas
rmsse_modelo_top_ventas: float = np.mean(rmsse_productos)
print(f"El RMSSE promedio del modelo top_ventas es {rmsse_modelo_top_ventas:.4f}")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios del cluster top_ventas
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados_top_ventas["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados_top_ventas[resultados_top_ventas["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} del cluster top_ventas")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se exportan los resultados
resultados_top_ventas.to_csv("res_xgboost_top_ventas.csv")

<a id="ej2.2"></a>
## 2.2. _Cluster residual_

En esta subapartado se entrena un modelo para el _cluster residual_. Para ello, primero se calculan las variables de ventana móvil de 28 días en el pasado.

In [ ]:
# Se extrae el grupo residual del dataset global
residual: pd.DataFrame = dataset_global[dataset_global["cluster_nombre"] == "residual"]

In [ ]:
# Se crean las nuevas variables para el conjunto de datos del grupo residual
HORIZONTE: int = max(residual["diasLeadtime"] + residual["diasEntrePedidos"])
residual_secuenciado: List[pd.DataFrame] = []

for producto, df_producto in residual.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    df_producto_sec: pd.DataFrame = secuenciador(
        df=df_producto,
        lag=LAG,
        horizonte=HORIZONTE,
        target="udsVenta"
    )
    
    df_producto_sec["producto"] = producto
    
    residual_secuenciado.append(df_producto_sec)

residual_secuenciado: pd.DataFrame = pd.concat(residual_secuenciado, ignore_index=True)

<br><br>Para entrenar el regresor _XGBoost_, se seleccionana todas las variables explicativas originales del _dataset_ global junto con los nuevos atributos de ventana móvil.

In [ ]:
# Se selecciona el subconjunto de variables de interés del conjunto de datos del cluster residual
residual_subset: pd.DataFrame = (
    residual_secuenciado[
        [
            "producto", "fecha", "diasEntrePedidos", "diasLeadtime", "eurPrecioMedio", "bolHoliday_reconstruido", "bolOpen_reconstruido",
            "isPromo", "lag_7", "lag_14", "lag_21", "lag_28", "dia_semana_str_Lunes", "dia_semana_str_Martes", "dia_semana_str_Miércoles",
            "dia_semana_str_Jueves", "dia_semana_str_Viernes", "dia_semana_str_Sábado", "dia_semana_str_Domingo", "media_rolling_7",
            "media_rolling_14", "media_rolling_21", "media_rolling_28", "max_rolling_14", "max_rolling_21", "max_rolling_28", "std_rolling_7",
            "std_rolling_14", "std_rolling_21", "std_rolling_28", "udsVenta"
        ] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)] + [f"mes_str_{i}" for i in range(1, 13)]
    ]
)

In [ ]:
# Se crean los conjuntos de entrenamiento y de prueba del cluster residual
horizontes: List[str] = [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

conjuntos_residual: Dict[str, pd.DataFrame] = {}
lista_x_train, lista_x_val, lista_x_test = [], [], []
lista_y_train, lista_y_val, lista_y_test = [], [], []

for producto, df_producto in residual_subset.drop("udsVenta", axis=1).groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").drop("producto", axis=1)
    
    x_producto: pd.DataFrame = df_producto.drop(columns=horizontes + ["fecha"])
    y_producto: pd.DataFrame = df_producto[horizontes]
    
    PARTICION_TRAIN_TEST: int = int(len(df_producto) * 0.8)
    datos_x_train: pd.DataFrame = x_producto.iloc[:PARTICION_TRAIN_TEST]
    lista_x_test.append(x_producto.iloc[PARTICION_TRAIN_TEST:])

    PARTICION_TRAIN_VAL: int = int(len(datos_x_train) * 0.8)
    lista_x_train.append(datos_x_train.iloc[:PARTICION_TRAIN_VAL])
    lista_x_val.append(datos_x_train.iloc[PARTICION_TRAIN_VAL:])
    

    datos_y_train: pd.Series = y_producto.iloc[:PARTICION_TRAIN_TEST]
    lista_y_test.append(y_producto.iloc[PARTICION_TRAIN_TEST:])
    
    lista_y_train.append(datos_y_train.iloc[:PARTICION_TRAIN_VAL])
    lista_y_val.append(datos_y_train.iloc[PARTICION_TRAIN_VAL:])
    

conjuntos_residual["x_train"] = pd.concat(lista_x_train, ignore_index=True)
conjuntos_residual["x_test"] = pd.concat(lista_x_test, ignore_index=True)
conjuntos_residual["x_val"] = pd.concat(lista_x_val, ignore_index=True)

conjuntos_residual["y_train"] = pd.concat(lista_y_train, ignore_index=True)
conjuntos_residual["y_test"]  = pd.concat(lista_y_test, ignore_index=True)
conjuntos_residual["y_val"]  = pd.concat(lista_y_val, ignore_index=True)

print(f"El conjunto x_train contiene {conjuntos_residual["x_train"].shape[0]} filas y {conjuntos_residual["x_train"].shape[1]} columnas")
print(f"El conjunto y_train contiene {conjuntos_residual["y_train"].shape[0]} filas y {conjuntos_residual["y_train"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_test contiene {conjuntos_residual["x_test"].shape[0]} filas y {conjuntos_residual["x_test"].shape[1]} columnas")
print(f"El conjunto y_test contiene {conjuntos_residual["y_test"].shape[0]} filas y {conjuntos_residual["y_test"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_val contiene {conjuntos_residual["x_val"].shape[0]} filas y {conjuntos_residual["x_val"].shape[1]} columnas")
print(f"El conjunto y_val contiene {conjuntos_residual["y_val"].shape[0]} filas y {conjuntos_residual["y_val"].shape[1]} columnas")

In [ ]:
# Se optimizan varios modelos XGBoost a lo largo de todo el horizonte de predicción del cluster residual
optimizaciones: List[int] = [i for i in range(1, HORIZONTE + 1, 13)]

mejores_max_depth: List[int] = []
mejores_learning_rate: List[float] = []
mejores_subsample: List[float] = []
mejores_colsample_bytree: List[float] = []
mejores_gamma: List[float] = []

tiempo_inicial_opt: float = time.time()
for h in optimizaciones:
    modelo: XGBRegressor = XGBRegressor(
        objective="reg:squarederror",
        n_jobs=-1
    )
    
    search_xgboost_residual: RandomizedSearchCV = RandomizedSearchCV(
        modelo,
        param_distributions=param_search_xgboost,
        n_iter=N_ITER,
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        verbose=1,
        random_state=SEMILLA
    )

    tiempo_inicial: float = time.time()
    search_xgboost_residual.fit(conjuntos_residual["x_train"], conjuntos_residual["y_train"][f"hor_{h}"])
    tiempo_final: float = time.time()
    
    mejores_max_depth.append(search_xgboost_residual.best_params_["max_depth"])
    mejores_learning_rate.append(search_xgboost_residual.best_params_["learning_rate"])
    mejores_subsample.append(search_xgboost_residual.best_params_["subsample"])
    mejores_colsample_bytree.append(search_xgboost_residual.best_params_["colsample_bytree"])
    mejores_gamma.append(search_xgboost_residual.best_params_["gamma"])

    
    print(
        "La primera iteración de la optimización de hiperparámetros de los modelos del cluster residual de "
        f"XGBoost tardó {round(tiempo_final - tiempo_inicial, 2)} segundos en completarse"
    )

tiempo_final_opt: float = time.time()
print("\n")
print(
    "La optimización de hiperparámetros de los modelos del cluster residual de XGBoost tardó "
    f"{round(tiempo_final_opt - tiempo_inicial_opt, 2)} segundos en completarse"
)

In [ ]:
# Se calculan los mejores hiperparámetros de las múltiples optimizaciones del cluster residual
max_depth_opt: int = moda(mejores_max_depth)
learning_rate_opt: float = round(np.mean(mejores_learning_rate), 2)
subsample_opt: float = round(np.mean(mejores_subsample), 1)
colsample_bytree_opt: float = round(np.mean(mejores_colsample_bytree), 1)
gamma_opt: float = round(np.mean(mejores_gamma), 2)

print(
    f"La profundidad máxima óptima de los árboles es de max_depth={max_depth_opt}, la mejor tasa de aprendizaje "
    f"es {learning_rate_opt}, la mejor fracción de muestras para entrenar a cada árbol es {subsample_opt}, la mejor "
    f"fracción de variables para entrenar cada árbol es {colsample_bytree_opt}, y la mejor reducción de pérdida mínima"
    f"para partir un nodo es {gamma_opt}"
)

In [ ]:
# Se entrenan los modelos XGBoost para predecir la demanda con los datos del cluster residual
modelos_residual: Dict[str, XGBRegressor] = {}
tiempo_inicial: int = time.time()

for h in range(1, HORIZONTE + 1):
    modelo_residual = XGBRegressor(
        max_depth=max_depth_opt,
        learning_rate=learning_rate_opt,
        subsample=subsample_opt,
        colsample_bytree=colsample_bytree_opt,
        gamma=gamma_opt,
        objective="reg:squarederror",
        eval_metric="rmse",
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=SEMILLA,
        n_jobs=-1
    )
    
    modelo_residual.fit(
        conjuntos_residual["x_train"],
        conjuntos_residual["y_train"][f"hor_{h}"],
        eval_set=[(
            conjuntos_residual["x_val"],
            conjuntos_residual["y_val"][f"hor_{h}"]
        )],
        verbose=False
    )
    
    modelos_residual[h] = modelo_residual

tiempo_final: int = time.time()
print(f"Los modelos del cluster residual tardaron {round(tiempo_final - tiempo_inicial, 2)} segundos en entrenarse")

In [ ]:
# Se realizan las predicciones en el conjunto de prueba del dataset del cluster residual
y_pred_residual: np.ndarray = np.column_stack([
    modelos_residual[h].predict(conjuntos_residual["x_test"])
    for h in range(1, HORIZONTE + 1)
])

y_pred_residual_df: pd.DataFrame = pd.DataFrame(
    y_pred_residual,
    columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
)

In [ ]:
# Se asocian a las predicciones y los valores reales los productos y las fechas correspondientes
lista_meta_test: List[pd.DataFrame] = []

for producto, df_producto in residual_subset.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    meta_info_test: pd.DataFrame = df_producto.iloc[PARTICION:][[
        "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
    ]]
    lista_meta_test.append(meta_info_test)

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)

y_real_residual_df: pd.DataFrame = pd.DataFrame(
    conjuntos_residual["y_test"].values,
    columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
)

resultados_residual: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_residual_df, y_pred_residual_df],
    axis=1
)

resultados_residual["fecha"] = pd.to_datetime(resultados_residual["fecha"])

In [ ]:
# Se calculan los errores cometidos en el conjunto de prueba del cluster residual
rmses: Dict[str, float] = {
    f"hor_{i}": rmse(
        resultados_residual[f"real_{i}"],
        resultados_residual[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Promedio del RMSE de los horizontes temporales del modelo del cluster residual")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<br><br>Se calculan los RMSE y RMSSE a nivel de horizonte por producto del _cluster residual_. Los RMSE se exportan en un archivo _csv_ para la **Evaluación del Impacto Económico**, y los RMSSE se usan para obtener el promedio del modelo de esta métrica como se explicó al inicio del _notebook_.

In [ ]:
# Se calculan los RMSE en el conjunto de prueba del cluster residual por producto y se exportan para la evaluación económica
rmses_residual: List[pd.DataFrame] = []

for producto, df_producto in resultados_residual.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_residual.append(rmses_df)

rmses_residual: pd.DataFrame = pd.concat(rmses_residual, ignore_index=True)
rmses_residual.to_csv("rmses_xgboost_residual.csv")

In [ ]:
# Se calculan los RMSSE de cada producto en cada horizonte temporal del modelo del grupo residual
lista_rmsse: List[Dict[str, Any]] = []
for horizonte in range(1, HORIZONTE + 1):

    rmsse_horizonte: Dict[str, float] = rmsse_producto(
        residual_subset,
        resultados_residual,
        horizonte
    )

    for producto, valor in rmsse_horizonte.items():

        lista_rmsse.append({
            "producto": producto,
            "horizonte": horizonte,
            "rmsse": valor
        })

# Se disponen todas las combinaciones de RMSSE de productos y horizontes en un DataFrame
df_rmsses: pd.DataFrame = pd.DataFrame(lista_rmsse)
df_rmsses = df_rmsses.pivot(
    index="producto",
    columns="horizonte",
    values="rmsse"
).reset_index()
df_rmsses.columns = ["producto"] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

# Se calcula el RMSSE de cada producto como el promedio de todos los RMSSE en cada horizonte temporal
# hasta el que corresponde con el ciclo de aprovisionamiento del artículo
rmsse_productos: List[float] = []
for producto, df_producto in resultados_residual.groupby("producto"):

    ciclo_aprov: int = int(
        df_producto["diasLeadtime"].mean() +
        df_producto["diasEntrePedidos"].mean()
    )

    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]

    fila_producto_df = df_rmsses[df_rmsses["producto"] == producto]
    if fila_producto_df.empty:
        continue
    fila_producto: pd.Series = fila_producto_df.iloc[0]

    rmsse_producto_promedio: float = fila_producto[columnas_horizontes].mean()
    rmsse_productos.append(rmsse_producto_promedio)

# Se promedian todos los RMSSEs de los ítems para obtener el RMSSE del modelo residual
rmsse_modelo_residual: float = np.mean(rmsse_productos)
print(f"El RMSSE promedio del modelo del cluster residual es {rmsse_modelo_residual:.4f}")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios del cluster residual
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados_residual["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados_residual[resultados_residual["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} del cluster residual")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se exportan los resultados
resultados_residual.to_csv("res_xgboost_residual.csv")

<a id="ej2.3"></a>
## 2.3. _Cluster alta_rotacion_

El siguiente _cluster_ que se utiliza para entrenar un conjunto de modelos _XGBoost_ es el grupo _alta_rotacion_, al cual se le aplica la función _**secuenciador()**_ para calcular las variables de ventana móvil.

In [ ]:
# Se extrae el grupo alta_rotacion del dataset global
alta_rotacion: pd.DataFrame = dataset_global[dataset_global["cluster_nombre"] == "alta_rotacion"]

In [ ]:
# Se crean las nuevas variables para el conjunto de datos del cluster alta_rotacion
HORIZONTE: int = max(alta_rotacion["diasLeadtime"] + alta_rotacion["diasEntrePedidos"])
alta_rotacion_secuenciado: List[pd.DataFrame] = []

for producto, df_producto in alta_rotacion.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    df_producto_sec: pd.DataFrame = secuenciador(
        df=df_producto,
        lag=LAG,
        horizonte=HORIZONTE,
        target="udsVenta"
    )
    
    df_producto_sec["producto"] = producto
    
    alta_rotacion_secuenciado.append(df_producto_sec)

alta_rotacion_secuenciado: pd.DataFrame = pd.concat(alta_rotacion_secuenciado, ignore_index=True)

<br><br>De la misma forma que en los otros _clusters_, se selecciona el subconjunto de variables de interés para crear los conjuntos de entrenamiento y de prueba con los que se entrenarán los modelos _XGBoost_.

In [ ]:
# Se selecciona el subconjunto de variables de interés del conjunto de datos de alta_rotacion
alta_rotacion_subset: pd.DataFrame = (
    alta_rotacion_secuenciado[
        [
            "producto", "fecha", "diasEntrePedidos", "diasLeadtime", "eurPrecioMedio", "bolHoliday_reconstruido", "bolOpen_reconstruido",
            "isPromo", "lag_7", "lag_14", "lag_21", "lag_28", "dia_semana_str_Lunes", "dia_semana_str_Martes", "dia_semana_str_Miércoles",
            "dia_semana_str_Jueves", "dia_semana_str_Viernes", "dia_semana_str_Sábado", "dia_semana_str_Domingo", "media_rolling_7",
            "media_rolling_14", "media_rolling_21", "media_rolling_28", "max_rolling_14", "max_rolling_21", "max_rolling_28", "std_rolling_7",
            "std_rolling_14", "std_rolling_21", "std_rolling_28", "udsVenta"
        ] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)] + [f"mes_str_{i}" for i in range(1, 13)]
    ]
)

In [ ]:
# Se crean los conjuntos de entrenamiento y de prueba del cluster alta_rotacion
horizontes: List[str] = [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

conjuntos_alta_rotacion: Dict[str, pd.DataFrame] = {}
lista_x_train, lista_x_val, lista_x_test = [], [], []
lista_y_train, lista_y_val, lista_y_test = [], [], []

for producto, df_producto in alta_rotacion_subset.drop("udsVenta", axis=1).groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").drop("producto", axis=1)
    
    x_producto: pd.DataFrame = df_producto.drop(columns=horizontes + ["fecha"])
    y_producto: pd.DataFrame = df_producto[horizontes]
    
    PARTICION_TRAIN_TEST: int = int(len(df_producto) * 0.8)
    datos_x_train: pd.DataFrame = x_producto.iloc[:PARTICION_TRAIN_TEST]
    lista_x_test.append(x_producto.iloc[PARTICION_TRAIN_TEST:])

    PARTICION_TRAIN_VAL: int = int(len(datos_x_train) * 0.8)
    lista_x_train.append(datos_x_train.iloc[:PARTICION_TRAIN_VAL])
    lista_x_val.append(datos_x_train.iloc[PARTICION_TRAIN_VAL:])
    

    datos_y_train: pd.Series = y_producto.iloc[:PARTICION_TRAIN_TEST]
    lista_y_test.append(y_producto.iloc[PARTICION_TRAIN_TEST:])
    
    lista_y_train.append(datos_y_train.iloc[:PARTICION_TRAIN_VAL])
    lista_y_val.append(datos_y_train.iloc[PARTICION_TRAIN_VAL:])
    

conjuntos_alta_rotacion["x_train"] = pd.concat(lista_x_train, ignore_index=True)
conjuntos_alta_rotacion["x_test"] = pd.concat(lista_x_test, ignore_index=True)
conjuntos_alta_rotacion["x_val"] = pd.concat(lista_x_val, ignore_index=True)

conjuntos_alta_rotacion["y_train"] = pd.concat(lista_y_train, ignore_index=True)
conjuntos_alta_rotacion["y_test"]  = pd.concat(lista_y_test, ignore_index=True)
conjuntos_alta_rotacion["y_val"]  = pd.concat(lista_y_val, ignore_index=True)

print(f"El conjunto x_train contiene {conjuntos_alta_rotacion["x_train"].shape[0]} filas y {conjuntos_alta_rotacion["x_train"].shape[1]} columnas")
print(f"El conjunto y_train contiene {conjuntos_alta_rotacion["y_train"].shape[0]} filas y {conjuntos_alta_rotacion["y_train"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_test contiene {conjuntos_alta_rotacion["x_test"].shape[0]} filas y {conjuntos_alta_rotacion["x_test"].shape[1]} columnas")
print(f"El conjunto y_test contiene {conjuntos_alta_rotacion["y_test"].shape[0]} filas y {conjuntos_alta_rotacion["y_test"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_val contiene {conjuntos_alta_rotacion["x_val"].shape[0]} filas y {conjuntos_alta_rotacion["x_val"].shape[1]} columnas")
print(f"El conjunto y_val contiene {conjuntos_alta_rotacion["y_val"].shape[0]} filas y {conjuntos_alta_rotacion["y_val"].shape[1]} columnas")

In [ ]:
# Se optimizan varios modelos XGBoost a lo largo de todo el horizonte de predicción del cluster alta_rotacion
optimizaciones: List[int] = [i for i in range(1, HORIZONTE + 1, 3)]

mejores_max_depth: List[int] = []
mejores_learning_rate: List[float] = []
mejores_subsample: List[float] = []
mejores_colsample_bytree: List[float] = []
mejores_gamma: List[float] = []

tiempo_inicial_opt: float = time.time()
for h in optimizaciones:
    modelo: XGBRegressor = XGBRegressor(
        objective="reg:squarederror",
        n_jobs=-1
    )
    
    search_xgboost_alta_rotacion: RandomizedSearchCV = RandomizedSearchCV(
        modelo,
        param_distributions=param_search_xgboost,
        n_iter=N_ITER,
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        verbose=1,
        random_state=SEMILLA
    )

    tiempo_inicial: float = time.time()
    search_xgboost_alta_rotacion.fit(conjuntos_alta_rotacion["x_train"], conjuntos_alta_rotacion["y_train"][f"hor_{h}"])
    tiempo_final: float = time.time()
    
    mejores_max_depth.append(search_xgboost_alta_rotacion.best_params_["max_depth"])
    mejores_learning_rate.append(search_xgboost_alta_rotacion.best_params_["learning_rate"])
    mejores_subsample.append(search_xgboost_alta_rotacion.best_params_["subsample"])
    mejores_colsample_bytree.append(search_xgboost_alta_rotacion.best_params_["colsample_bytree"])
    mejores_gamma.append(search_xgboost_alta_rotacion.best_params_["gamma"])

    
    print(
        "La primera iteración de la optimización de hiperparámetros de los modelos de alta_rotacion de XGBoost tardó "
        f"{round(tiempo_final - tiempo_inicial, 2)} segundos en completarse"
    )

tiempo_final_opt: float = time.time()
print("\n")
print(
    "La optimización de hiperparámetros de los modelos de alta_rotacion de XGBoost tardó "
    f"{round(tiempo_final_opt - tiempo_inicial_opt, 2)} segundos en completarse"
)

In [ ]:
# Se calculan los mejores hiperparámetros de las múltiples optimizaciones del cluster alta_rotacion
max_depth_opt: int = moda(mejores_max_depth)
learning_rate_opt: float = round(np.mean(mejores_learning_rate), 2)
subsample_opt: float = round(np.mean(mejores_subsample), 1)
colsample_bytree_opt: float = round(np.mean(mejores_colsample_bytree), 1)
gamma_opt: float = round(np.mean(mejores_gamma), 2)

print(
    f"La profundidad máxima óptima de los árboles es de max_depth={max_depth_opt}, la mejor tasa de aprendizaje "
    f"es {learning_rate_opt}, la mejor fracción de muestras para entrenar a cada árbol es {subsample_opt}, la mejor "
    f"fracción de variables para entrenar cada árbol es {colsample_bytree_opt}, y la mejor reducción de pérdida mínima"
    f"para partir un nodo es {gamma_opt}"
)

In [ ]:
# Se entrenan los modelos XGBoost para predecir la demanda con los datos del cluster alta_rotacion
modelos_alta_rotacion: Dict[str, XGBRegressor] = {}
tiempo_inicial: int = time.time()

for h in range(1, HORIZONTE + 1):
    modelo_alta_rotacion = XGBRegressor(
        max_depth=max_depth_opt,
        learning_rate=learning_rate_opt,
        subsample=subsample_opt,
        colsample_bytree=colsample_bytree_opt,
        gamma=gamma_opt,
        objective="reg:squarederror",
        eval_metric="rmse",
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=SEMILLA,
        n_jobs=-1
    )
    
    modelo_alta_rotacion.fit(
        conjuntos_alta_rotacion["x_train"],
        conjuntos_alta_rotacion["y_train"][f"hor_{h}"],
        eval_set=[(
            conjuntos_alta_rotacion["x_val"],
            conjuntos_alta_rotacion["y_val"][f"hor_{h}"]
        )],
        verbose=False
    )
    
    modelos_alta_rotacion[h] = modelo_alta_rotacion

tiempo_final: int = time.time()
print(f"Los modelos alta_rotacion tardaron {round(tiempo_final - tiempo_inicial, 2)} segundos en entrenarse")

In [ ]:
# Se realizan las predicciones en el conjunto de prueba del dataset de alta_rotacion
y_pred_alta_rotacion: np.ndarray = np.column_stack([
    modelos_alta_rotacion[h].predict(conjuntos_alta_rotacion["x_test"])
    for h in range(1, HORIZONTE + 1)
])

y_pred_alta_rotacion_df: pd.DataFrame = pd.DataFrame(
    y_pred_alta_rotacion,
    columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
)

In [ ]:
# Se asocian a las predicciones y los valores reales los productos y las fechas correspondientes
lista_meta_test: List[pd.DataFrame] = []

for producto, df_producto in alta_rotacion_subset.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    meta_info_test: pd.DataFrame = df_producto.iloc[PARTICION:][[
        "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
    ]]
    lista_meta_test.append(meta_info_test)

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)

y_real_alta_rotacion_df: pd.DataFrame = pd.DataFrame(
    conjuntos_alta_rotacion["y_test"].values,
    columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
)

resultados_alta_rotacion: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_alta_rotacion_df, y_pred_alta_rotacion_df],
    axis=1
)

resultados_alta_rotacion["fecha"] = pd.to_datetime(resultados_alta_rotacion["fecha"])

In [ ]:
# Se calculan los errores cometidos en el conjunto de prueba del cluster alta_rotacion
rmses: Dict[str, float] = {
    f"hor_{i}": rmse(
        resultados_alta_rotacion[f"real_{i}"],
        resultados_alta_rotacion[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Promedio del RMSE de los horizontes temporales del modelo alta_rotacion")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<br><br>Se calculan los RMSE y RMSSE a nivel de horizonte por producto del _cluster alta_rotacion_. Los RMSE se exportan en un archivo _csv_ para la **Evaluación del Impacto Económico**, y los RMSSE se usan para obtener el promedio del modelo de esta métrica como se explicó al inicio del _notebook_.

In [ ]:
# Se calculan los RMSE en el conjunto de prueba del cluster alta_rotacion por producto y se exportan para la evaluación económica
rmses_alta_rotacion: List[pd.DataFrame] = []

for producto, df_producto in resultados_alta_rotacion.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_alta_rotacion.append(rmses_df)

rmses_alta_rotacion: pd.DataFrame = pd.concat(rmses_alta_rotacion, ignore_index=True)
rmses_alta_rotacion.to_csv("rmses_xgboost_alta_rotacion.csv")

In [ ]:
# Se calculan los RMSSE de cada producto en cada horizonte temporal del modelo del grupo alta_rotacion
lista_rmsse: List[Dict[str, Any]] = []
for horizonte in range(1, HORIZONTE + 1):

    rmsse_horizonte: Dict[str, float] = rmsse_producto(
        alta_rotacion_subset,
        resultados_alta_rotacion,
        horizonte
    )

    for producto, valor in rmsse_horizonte.items():

        lista_rmsse.append({
            "producto": producto,
            "horizonte": horizonte,
            "rmsse": valor
        })

# Se disponen todas las combinaciones de RMSSE de productos y horizontes en un DataFrame
df_rmsses: pd.DataFrame = pd.DataFrame(lista_rmsse)
df_rmsses = df_rmsses.pivot(
    index="producto",
    columns="horizonte",
    values="rmsse"
).reset_index()
df_rmsses.columns = ["producto"] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

# Se calcula el RMSSE de cada producto como el promedio de todos los RMSSE en cada horizonte temporal
# hasta el que corresponde con el ciclo de aprovisionamiento del artículo
rmsse_productos: List[float] = []
for producto, df_producto in resultados_alta_rotacion.groupby("producto"):

    ciclo_aprov: int = int(
        df_producto["diasLeadtime"].mean() +
        df_producto["diasEntrePedidos"].mean()
    )

    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]

    fila_producto_df = df_rmsses[df_rmsses["producto"] == producto]
    if fila_producto_df.empty:
        continue
    fila_producto: pd.Series = fila_producto_df.iloc[0]

    rmsse_producto_promedio: float = fila_producto[columnas_horizontes].mean()
    rmsse_productos.append(rmsse_producto_promedio)

# Se promedian todos los RMSSEs de los ítems para obtener el RMSSE del modelo alta_rotacion
rmsse_modelo_alta_rotacion: float = np.mean(rmsse_productos)
print(f"El RMSSE promedio del modelo alta_rotacion es {rmsse_modelo_alta_rotacion:.4f}")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios del cluster alta_rotacion
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados_alta_rotacion["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados_alta_rotacion[resultados_alta_rotacion["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} del cluster alta_rotacion")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se exportan los resultados
resultados_alta_rotacion.to_csv("res_xgboost_alta_rotacion.csv")

<a id="ej2.4"></a>
## 2.4. _Cluster estandar_

Finalmente, en esta sección se extraen los datos del _cluster estandar_ del conjunto de datos global para entrenar un regresor _XGBoost_.

In [ ]:
# Se extrae el cluster estandar del dataset global
estandar: pd.DataFrame = dataset_global[dataset_global["cluster_nombre"] == "estandar"]

In [ ]:
# Se crean las nuevas variables para el conjunto de datos del cluster estandar
HORIZONTE: int = max(estandar["diasLeadtime"] + estandar["diasEntrePedidos"])
estandar_secuenciado: List[pd.DataFrame] = []

for producto, df_producto in estandar.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    df_producto_sec: pd.DataFrame = secuenciador(
        df=df_producto,
        lag=LAG,
        horizonte=HORIZONTE,
        target="udsVenta"
    )
    
    df_producto_sec["producto"] = producto
    
    estandar_secuenciado.append(df_producto_sec)

estandar_secuenciado: pd.DataFrame = pd.concat(estandar_secuenciado, ignore_index=True)

<br><br>Los modelos se entrenan con el siguiente subconjunto de variables del _cluster estandar_.

In [ ]:
# Se selecciona el subconjunto de variables de interés del conjunto de datos del cluster estandar
estandar_subset: pd.DataFrame = (
    estandar_secuenciado[
        [
            "producto", "fecha", "diasEntrePedidos", "diasLeadtime", "eurPrecioMedio", "bolHoliday_reconstruido", "bolOpen_reconstruido",
            "isPromo", "lag_7", "lag_14", "lag_21", "lag_28", "dia_semana_str_Lunes", "dia_semana_str_Martes", "dia_semana_str_Miércoles",
            "dia_semana_str_Jueves", "dia_semana_str_Viernes", "dia_semana_str_Sábado", "dia_semana_str_Domingo", "media_rolling_7",
            "media_rolling_14", "media_rolling_21", "media_rolling_28", "max_rolling_14", "max_rolling_21", "max_rolling_28", "std_rolling_7",
            "std_rolling_14", "std_rolling_21", "std_rolling_28", "udsVenta"
        ] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)] + [f"mes_str_{i}" for i in range(1, 13)]
    ]
)

In [ ]:
# Se crean los conjuntos de entrenamiento y de prueba del cluster estandar
horizontes: List[str] = [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

conjuntos_estandar: Dict[str, pd.DataFrame] = {}
lista_x_train, lista_x_val, lista_x_test = [], [], []
lista_y_train, lista_y_val, lista_y_test = [], [], []

for producto, df_producto in estandar_subset.drop("udsVenta", axis=1).groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values(by="fecha").drop("producto", axis=1)
    
    x_producto: pd.DataFrame = df_producto.drop(columns=horizontes + ["fecha"])
    y_producto: pd.DataFrame = df_producto[horizontes]
    
    PARTICION_TRAIN_TEST: int = int(len(df_producto) * 0.8)
    datos_x_train: pd.DataFrame = x_producto.iloc[:PARTICION_TRAIN_TEST]
    lista_x_test.append(x_producto.iloc[PARTICION_TRAIN_TEST:])

    PARTICION_TRAIN_VAL: int = int(len(datos_x_train) * 0.8)
    lista_x_train.append(datos_x_train.iloc[:PARTICION_TRAIN_VAL])
    lista_x_val.append(datos_x_train.iloc[PARTICION_TRAIN_VAL:])
    

    datos_y_train: pd.Series = y_producto.iloc[:PARTICION_TRAIN_TEST]
    lista_y_test.append(y_producto.iloc[PARTICION_TRAIN_TEST:])
    
    lista_y_train.append(datos_y_train.iloc[:PARTICION_TRAIN_VAL])
    lista_y_val.append(datos_y_train.iloc[PARTICION_TRAIN_VAL:])
    

conjuntos_estandar["x_train"] = pd.concat(lista_x_train, ignore_index=True)
conjuntos_estandar["x_test"] = pd.concat(lista_x_test, ignore_index=True)
conjuntos_estandar["x_val"] = pd.concat(lista_x_val, ignore_index=True)

conjuntos_estandar["y_train"] = pd.concat(lista_y_train, ignore_index=True)
conjuntos_estandar["y_test"]  = pd.concat(lista_y_test, ignore_index=True)
conjuntos_estandar["y_val"]  = pd.concat(lista_y_val, ignore_index=True)

print(f"El conjunto x_train contiene {conjuntos_estandar["x_train"].shape[0]} filas y {conjuntos_estandar["x_train"].shape[1]} columnas")
print(f"El conjunto y_train contiene {conjuntos_estandar["y_train"].shape[0]} filas y {conjuntos_estandar["y_train"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_test contiene {conjuntos_estandar["x_test"].shape[0]} filas y {conjuntos_estandar["x_test"].shape[1]} columnas")
print(f"El conjunto y_test contiene {conjuntos_estandar["y_test"].shape[0]} filas y {conjuntos_estandar["y_test"].shape[1]} columnas")
print("\n")
print(f"El conjunto x_val contiene {conjuntos_estandar["x_val"].shape[0]} filas y {conjuntos_estandar["x_val"].shape[1]} columnas")
print(f"El conjunto y_val contiene {conjuntos_estandar["y_val"].shape[0]} filas y {conjuntos_estandar["y_val"].shape[1]} columnas")

In [ ]:
# Se optimizan varios modelos XGBoost a lo largo de todo el horizonte de predicción del cluster estandar
optimizaciones: List[int] = [i for i in range(1, HORIZONTE + 1, 7)]

mejores_max_depth: List[int] = []
mejores_learning_rate: List[float] = []
mejores_subsample: List[float] = []
mejores_colsample_bytree: List[float] = []
mejores_gamma: List[float] = []

tiempo_inicial_opt: float = time.time()
for h in optimizaciones:
    modelo: XGBRegressor = XGBRegressor(
        objective="reg:squarederror",
        n_jobs=-1
    )
    
    search_xgboost_estandar: RandomizedSearchCV = RandomizedSearchCV(
        modelo,
        param_distributions=param_search_xgboost,
        n_iter=N_ITER,
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        verbose=1,
        random_state=SEMILLA
    )

    tiempo_inicial: float = time.time()
    search_xgboost_estandar.fit(conjuntos_estandar["x_train"], conjuntos_estandar["y_train"][f"hor_{h}"])
    tiempo_final: float = time.time()
    
    mejores_max_depth.append(search_xgboost_estandar.best_params_["max_depth"])
    mejores_learning_rate.append(search_xgboost_estandar.best_params_["learning_rate"])
    mejores_subsample.append(search_xgboost_estandar.best_params_["subsample"])
    mejores_colsample_bytree.append(search_xgboost_estandar.best_params_["colsample_bytree"])
    mejores_gamma.append(search_xgboost_estandar.best_params_["gamma"])

    
    print(
        "La primera iteración de la optimización de hiperparámetros de los modelos del cluster estandar "
        f"de XGBoost tardó {round(tiempo_final - tiempo_inicial, 2)} segundos en completarse"
    )

tiempo_final_opt: float = time.time()
print("\n")
print(
    "La optimización de hiperparámetros de los modelos del cluster estandar de XGBoost tardó "
    f"{round(tiempo_final_opt - tiempo_inicial_opt, 2)} segundos en completarse"
)

In [ ]:
# Se calculan los mejores hiperparámetros de las múltiples optimizaciones del cluster estandar
max_depth_opt: int = moda(mejores_max_depth)
learning_rate_opt: float = round(np.mean(mejores_learning_rate), 2)
subsample_opt: float = round(np.mean(mejores_subsample), 1)
colsample_bytree_opt: float = round(np.mean(mejores_colsample_bytree), 1)
gamma_opt: float = round(np.mean(mejores_gamma), 2)

print(
    f"La profundidad máxima óptima de los árboles es de max_depth={max_depth_opt}, la mejor tasa de aprendizaje "
    f"es {learning_rate_opt}, la mejor fracción de muestras para entrenar a cada árbol es {subsample_opt}, la mejor "
    f"fracción de variables para entrenar cada árbol es {colsample_bytree_opt}, y la mejor reducción de pérdida mínima"
    f"para partir un nodo es {gamma_opt}"
)

In [ ]:
# Se entrenan los modelos XGBoost para predecir la demanda con los datos del cluster estandar
modelos_estandar: Dict[str, XGBRegressor] = {}
tiempo_inicial: int = time.time()

for h in range(1, HORIZONTE + 1):
    modelo_estandar = XGBRegressor(
        max_depth=max_depth_opt,
        learning_rate=learning_rate_opt,
        subsample=subsample_opt,
        colsample_bytree=colsample_bytree_opt,
        gamma=gamma_opt,
        objective="reg:squarederror",
        eval_metric="rmse",
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=SEMILLA,
        n_jobs=-1
    )
    
    modelo_estandar.fit(
        conjuntos_estandar["x_train"],
        conjuntos_estandar["y_train"][f"hor_{h}"],
        eval_set=[(
            conjuntos_estandar["x_val"],
            conjuntos_estandar["y_val"][f"hor_{h}"]
        )],
        verbose=False
    )
    
    modelos_estandar[h] = modelo_estandar

tiempo_final: int = time.time()
print(f"Los modelos del cluster estandar tardaron {round(tiempo_final - tiempo_inicial, 2)} segundos en entrenarse")

In [ ]:
# Se realizan las predicciones en el conjunto de prueba del dataset del cluster estandar
y_pred_estandar: np.ndarray = np.column_stack([
    modelos_estandar[h].predict(conjuntos_estandar["x_test"])
    for h in range(1, HORIZONTE + 1)
])

y_pred_estandar_df: pd.DataFrame = pd.DataFrame(
    y_pred_estandar,
    columns=[f"pred_{h}" for h in range(1, HORIZONTE + 1)]
)

In [ ]:
# Se asocian a las predicciones y los valores reales los productos y las fechas correspondientes
lista_meta_test: List[pd.DataFrame] = []

for producto, df_producto in estandar_subset.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")
    
    PARTICION: int = int(len(df_producto) * 0.8)
    
    meta_info_test: pd.DataFrame = df_producto.iloc[PARTICION:][[
        "producto", "fecha", "udsVenta", "diasLeadtime", "diasEntrePedidos", "eurPrecioMedio"
    ]]
    lista_meta_test.append(meta_info_test)

meta_test: pd.DataFrame = pd.concat(lista_meta_test, ignore_index=True)

y_real_estandar_df: pd.DataFrame = pd.DataFrame(
    conjuntos_estandar["y_test"].values,
    columns=[f"real_{h}" for h in range(1, HORIZONTE + 1)]
)

resultados_estandar: pd.DataFrame = pd.concat(
    [meta_test.reset_index(drop=True), y_real_estandar_df, y_pred_estandar_df],
    axis=1
)

resultados_estandar["fecha"] = pd.to_datetime(resultados_estandar["fecha"])

In [ ]:
# Se calculan los errores cometidos en el conjunto de prueba del cluster estandar
rmses: Dict[str, float] = {
    f"hor_{i}": rmse(
        resultados_estandar[f"real_{i}"],
        resultados_estandar[f"pred_{i}"]
    ) for i in range(1, HORIZONTE + 1)
}

In [ ]:
# Se visualiza la evolución del promedio del RMSE de todos los productos en los horizontes temporales
plt.plot(range(1, HORIZONTE + 1, 1), rmses.values(), color="blue")

plt.title("Promedio del RMSE de los horizontes temporales del modelo del cluster estandar")
plt.xlabel("Días futuros")
plt.xticks(range(1, HORIZONTE + 1, 5))
plt.ylabel("RMSE promedio por producto")

plt.grid(True, alpha=0.3)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<br><br>Se calculan los RMSE y RMSSE a nivel de horizonte por producto del _cluster estandar_. Los RMSE se exportan en un archivo _csv_ para la **Evaluación del Impacto Económico**, y los RMSSE se usan para obtener el promedio del modelo de esta métrica como se explicó al inicio del _notebook_.

In [ ]:
# Se calculan los RMSE en el conjunto de prueba del cluster estandar por producto y se exportan para la evaluación económica
rmses_estandar: List[pd.DataFrame] = []

for producto, df_producto in resultados_estandar.groupby("producto"):
    df_producto: pd.DataFrame = df_producto.sort_values("fecha")

    rmses_producto: Dict[str, float] = {
        f"hor_{i}": [rmse(
            df_producto[f"real_{i}"],
            df_producto[f"pred_{i}"]
        )] for i in range(1, HORIZONTE + 1)
    }

    rmses_df: pd.DataFrame = pd.DataFrame(rmses_producto)
    rmses_df["producto"] = producto
    rmses_estandar.append(rmses_df)

rmses_estandar: pd.DataFrame = pd.concat(rmses_estandar, ignore_index=True)
rmses_estandar.to_csv("rmses_xgboost_estandar.csv")

In [ ]:
# Se calculan los RMSSE de cada producto en cada horizonte temporal del modelo del cluster estandar
lista_rmsse: List[Dict[str, Any]] = []
for horizonte in range(1, HORIZONTE + 1):

    rmsse_horizonte: Dict[str, float] = rmsse_producto(
        estandar_subset,
        resultados_estandar,
        horizonte
    )

    for producto, valor in rmsse_horizonte.items():

        lista_rmsse.append({
            "producto": producto,
            "horizonte": horizonte,
            "rmsse": valor
        })

# Se disponen todas las combinaciones de RMSSE de productos y horizontes en un DataFrame
df_rmsses: pd.DataFrame = pd.DataFrame(lista_rmsse)
df_rmsses = df_rmsses.pivot(
    index="producto",
    columns="horizonte",
    values="rmsse"
).reset_index()
df_rmsses.columns = ["producto"] + [f"hor_{i}" for i in range(1, HORIZONTE + 1)]

# Se calcula el RMSSE de cada producto como el promedio de todos los RMSSE en cada horizonte temporal
# hasta el que corresponde con el ciclo de aprovisionamiento del artículo
rmsse_productos: List[float] = []
for producto, df_producto in resultados_estandar.groupby("producto"):

    ciclo_aprov: int = int(
        df_producto["diasLeadtime"].mean() +
        df_producto["diasEntrePedidos"].mean()
    )

    columnas_horizontes: List[str] = [f"hor_{h}" for h in range(1, ciclo_aprov + 1)]

    fila_producto_df = df_rmsses[df_rmsses["producto"] == producto]
    if fila_producto_df.empty:
        continue
    fila_producto: pd.Series = fila_producto_df.iloc[0]

    rmsse_producto_promedio: float = fila_producto[columnas_horizontes].mean()
    rmsse_productos.append(rmsse_producto_promedio)

# Se promedian todos los RMSSEs de los ítems para obtener el RMSSE del modelo del cluster estandar
rmsse_modelo_estandar: float = np.mean(rmsse_productos)
print(f"El RMSSE promedio del modelo del cluster estandar es {rmsse_modelo_estandar:.4f}")

In [ ]:
# Se visualiza la demanda real frente a las predicciones de tres productos aleatorios del cluster estandar
N_PRODUCTOS: int = 3
N_PREDICCIONES: int = 3
seleccion_productos: List[int] = random.sample(resultados_estandar["producto"].unique().tolist(), N_PRODUCTOS)

fig, axs = plt.subplots(nrows=len(seleccion_productos), ncols=1, figsize=(10, 20))

for ax, producto in zip(axs, seleccion_productos):
    res_producto: pd.DataFrame = resultados_estandar[resultados_estandar["producto"] == producto]
    seleccion_predicciones: List[int] = random.sample(range(len(res_producto)), N_PREDICCIONES)

    fecha_max: str = res_producto["fecha"].max()
    indices_validos: List[int] = [
        i for i in range(len(res_producto))
        if res_producto.iloc[i]["fecha"] + pd.Timedelta(days=HORIZONTE) <= fecha_max
    ]
    seleccion_predicciones: List[int] = random.sample(indices_validos, N_PREDICCIONES)
    
    fechas_real: pd.Series = res_producto["fecha"] + pd.Timedelta(days=1)
    ax.plot(fechas_real, res_producto["real_1"], label="demanda_real", color="blue", alpha=0.5)

    for i, indice_prediccion in enumerate(seleccion_predicciones):
        fila: pd.Series = res_producto.iloc[indice_prediccion]
        predicciones = [fila[f"pred_{h}"] for h in range(1, HORIZONTE + 1)]

        fecha_inicio: str = fila["fecha"]
        fechas_prediccion = [fecha_inicio + pd.Timedelta(days=h) for h in range(1, HORIZONTE + 1)]
        
        ax.plot(fechas_prediccion, predicciones, label=f"prediccion_{i + 1}", linestyle="--", alpha=0.5)

    ax.set_title(f"Demanda real Vs Predicciones de la demanda del producto {producto} del cluster estandar")
    ax.set_xlabel("Días")
    ax.set_xticks([res_producto["fecha"].iloc[i] for i in range(0, len(res_producto["fecha"]), 20)])
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Unidades")
    
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Se exportan los resultados
resultados_estandar.to_csv("res_xgboost_estandar.csv")